## 1. Read data

### 1.1. The dataset of sentences

In [1]:
import pandas as pd
sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/analysis_datasets/translated_sentence_data.xlsx")

In [2]:
sentences = sentence_df["Clean_Sentence"].to_list()

### 1.2. Import list of stopwords

In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/analysis_datasets/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec'
]

dutch_stopwords.extend(extra_list)

## 2. Download the embedding model BERTje

### 2.1 Download BERTje

In [4]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 2.2 Pre-calculate and save sentence embeddings (skip if there are saved embeddings and go to 2.3)

In [ ]:
# pre-calculate the embeddings of the dutch sentences
import numpy as np

embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])

In [ ]:
# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/saved_models/sentence_embeddings_bertje_nl.npy', sentence_embeddings_nl)

### 2.3 Load saved embeddings

In [5]:
# load the save model
import numpy as np

loaded_embeddings_nl = np.load('/workspace/mijnidbcoachnlp/saved_models/sentence_embeddings_bertje_nl.npy')

In [6]:
embeddings=loaded_embeddings_nl

## 3. Preparation for Fine-Tuning Hyperparameters

### 3.1. Generate a list of hyperparameter combinations

In [11]:
import itertools

# Generating a list of hyperparameter combinations
# UMAP hyperparameters
n_neighbors_values = [5, 10, 15, 20, 25, 30] # we are aiming for strict clusters so the n_neighbors range is relatively small
n_components_values = [2, 3, 4, 5]

# HDBSCAN hyperparameters
min_cluster_size_values = [10, 15, 20, 30]
min_samples_values = [5, 10, 15, 20, 30]

combinations = list(itertools.product(n_neighbors_values, n_components_values, min_cluster_size_values, min_samples_values))

# set min_samples < min_cluster_size
filtered_combinations = [
    combination for combination in combinations if combination[3] <= combination[2] 
]

# print the total number of combinations
len(filtered_combinations)

336

### 3.2 Function to calculate intra-topic similarity

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# function to calculate intra-topic cosine similarity
def calculate_intra_topic_cosine_similarity(embeddings, doc_topics):
    topic_similarities = []
    
    for topic in set(doc_topics):
        if topic == -1:  # Skip the "outlier" topic
            continue
        
        # Get embeddings of documents in the current topic
        topic_embeddings = embeddings[np.array(doc_topics) == topic]
        
        if len(topic_embeddings) < 2:
            continue  # Skip topics with a single document
        
        # Calculate cosine similarities and average them
        cosine_sim = cosine_similarity(topic_embeddings)
        avg_cosine_sim = cosine_sim[np.triu_indices_from(cosine_sim, k=1)].mean()
        
        topic_similarities.append(avg_cosine_sim)

    return topic_similarities

### 3.3. Function to calculate silhouette scores

In [11]:
# function to calculate silhouette scores if needed (we don't use the silhouette scores for now because the clusters are probably overlapping a lot so inter-topic distance is not so informative)
from sklearn.metrics import silhouette_samples
import numpy as np

def calculate_intra_topic_silhouette_score(embeddings, topics):

    # Filter out documents assigned to the outlier topic (-1)
    mask = topics != -1
    filtered_embeddings = embeddings[mask]
    filtered_topics = topics[mask]

    # Compute silhouette scores
    silhouette_scores = silhouette_samples(filtered_embeddings, filtered_topics, metric="cosine")
    
    # Calculate average silhouette score for each topic
    unique_topics = np.unique(filtered_topics)
    avg_silhouette_per_topic = {
        topic: np.mean(silhouette_scores[filtered_topics == topic]) for topic in unique_topics
    }

    # Compute the overall average silhouette score across all topics
    avg_silhouette_score = np.mean(list(avg_silhouette_per_topic.values()))

    return avg_silhouette_score


### 3.4 Function to calculate coherence scores

In [12]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 199.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
# create the dictionary of documents
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_dictionary(documents):

    processed_docs = [doc.split() for doc in documents]
    
    # Create a Gensim dictionary
    dictionary = Dictionary(processed_docs)
    
    return dictionary, processed_docs

dictionary, processed_docs = generate_dictionary(sentences)

In [14]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_topic_words(topics):
    topic_words = []
    
    for topic_num, words in topics.items():
        if topic_num == -1:  # Skip the outlier topic
            continue
        # Collect only the top N words for each topic
        top_words = [word for word, _ in words[:top_n_words]]
        topic_words.append(top_words)
    
    return topic_words

def calculate_coherence_score(topics, dictionary, processed_docs, coherence='c_v', top_n_words=10):

    topic_words = generate_topic_words(topics)

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=processed_docs,
        dictionary=dictionary,
        coherence=coherence
    )
    
    # Step 4: Get the coherence score
    coherence_score = coherence_model.get_coherence()
    
    return coherence_score


## 4. Fine-Tune BERTje-HDBSCAN model

### 4.1 Fine-Tuning UMAP hyper parameters

In [10]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 2), token_pattern=r'\b[a-zA-Z]{2,}\b')


### 4.2 Function to build BERTopic model using a combination of hyperparameters

In [23]:
# build the BERTopic model
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Define a function to process each combination
def process_combination(combination):
    n_neighbors, n_components, min_cluster_size, min_samples = combination
    
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=n_neighbors, n_components=n_components, metric='cosine', random_state=42),
        hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric='euclidean', cluster_selection_method='eom', prediction_data=True),
        vectorizer_model=vectorizer_model,
        nr_topics=50,
        top_n_words=10,
        verbose=True
    )

    # Train model
    doc_topics, probs = topic_model.fit_transform(sentences, embeddings)
    topics = topic_model.get_topics()
    
    
    # Compute metrics
    intra_topic_similarity = calculate_intra_topic_cosine_similarity(embeddings, doc_topics)
    coherence_score = calculate_coherence_score(topics, dictionary, processed_docs, coherence='c_v', top_n_words=10)
    return combination, intra_topic_similarity, coherence_score


### 4.3 Multiprocess for fine-tuning

In [15]:
# function to divide inputs in chunks
def chunk_list(data, chunk_size):
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

In [16]:
# divide filtered_combinations in batches
batch_size = 20

batches = list(chunk_list(filtered_combinations, batch_size))

len(batches) # there are 17 batches

17

In [19]:
# function to save intermediate batch results
import pickle
import os

def save_batch(batch_results, batch_index):
    filename = f"/workspace/mijnidbcoachnlp/result_data/batch_{batch_index}.pkl"
    with open(filename, 'wb') as file:
        pickle.dump(batch_results, file)
    print(f"Batch {batch_index} saved.")


In [24]:
import multiprocessing
import os

# Resume from the last saved batch
processed_batches = [
    int(f.split('_')[1].split('.')[0]) 
    for f in os.listdir("/workspace/mijnidbcoachnlp/result_data/") 
    if f.startswith("batch_") and f.endswith(".pkl")
]
start_batch = max(processed_batches) + 1 if processed_batches else 0

num_cores = 32
    
with multiprocessing.Pool(processes=num_cores) as pool:
    for batch_index, batch in enumerate(batches[start_batch:], start=start_batch):
        batch_results = pool.map(process_combination, batch)
        
        # Save each batch individually
        save_batch(batch_results, batch_index)

2024-11-06 11:37:41,619 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,619 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,623 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,625 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,626 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,625 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,627 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,628 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,628 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-06 11:37:41,632 - BERTopic - Dimensionality - Fitting the dimensionality reduction 

KeyboardInterrupt: 

In [25]:
# Load all batches into a single results list
all_results = []
for batch_index in range(len(batches)):
    with open(f"batch_{batch_index}.pkl", 'rb') as file:
        batch_results = pickle.load(file)
        all_results.extend(batch_results)

# Optionally separate into result_combinations and model_topic_similarities
result_combinations, model_topic_similarities, model_coherence_scores = zip(*all_results)

FileNotFoundError: [Errno 2] No such file or directory: 'batch_0.pkl'